# Feature Story 5 — Supervised classification with scikit-learn Pipelines

Predict a new customer's segment (`target`) from the 7 raw behavioural variables.

**Starting point:** `df_clean` values + `target` only — no clustering columns (`cluster_kmeans`, `cluster_dbscan`, …), and no log1p / scaler / PCA outside the pipeline. Those treatments belong to the clustering branch; applying them here before the split would leak information from the test set.

**Vigilance:** preprocessing lives *inside* each `Pipeline`, so `StandardScaler` is re-learned on every training fold and only ever applied (never fitted) on test data — one object, same steps at train and predict time.

In [ ]:
%matplotlib inline
import time
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     StratifiedKFold, GridSearchCV)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, ConfusionMatrixDisplay)
from imblearn.over_sampling import SMOTE

df = pd.read_csv("../../data/processed/df_clean_clustered.csv")
feats = ["BALANCE", "PURCHASES", "ONEOFF_PURCHASES",
         "INSTALLMENTS_PURCHASES", "CASH_ADVANCE", "CREDIT_LIMIT", "PAYMENTS"]
X = df[feats]
y = df["target"]

print(X.shape, "| target classes:", sorted(y.unique()))


### Class balance

The brief asks for resampling (`imblearn`) *if needed*. The three segments sit at 37.9 / 27.1 / 35.0% — close enough to balanced, so nothing is broken. SMOTE is still applied right before training as the brief's class-imbalance option; scores are expected to stay roughly the same, which is normal at this balance.

In [ ]:
counts = pd.DataFrame({
    "n": y.value_counts(),
    "share": (y.value_counts(normalize=True) * 100).round(1).astype(str) + "%",
})
print(counts.to_string())


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y)
print("train:", X_train.shape, " test:", X_test.shape)


### SMOTE — balance the training set

SMOTE creates synthetic customers in the minority classes so the models train on a balanced set (`imblearn`, the class-imbalance option the brief allows).

Two rules: SMOTE sees **training data only** — the test set stays untouched and real, so the scores below stay honest — and it runs *just before training*, so every model below learns from the resampled `X_train`.

In [ ]:
smote = SMOTE(random_state=0)
print("train before SMOTE:")
print(y_train.value_counts().to_string())
print()

X_train, y_train = smote.fit_resample(X_train, y_train)

print("train after SMOTE:")
print(y_train.value_counts().to_string())


### One object per model

`Pipeline([("scaler", StandardScaler()), ("classifier", …)])` — the four pipelines below differ **only** in their `classifier` step. Split, scaling and metrics stay identical, so the comparison is fair.

In [ ]:
pipelines = {
    "LogisticRegression": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=1000))]),
    "DecisionTree": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", DecisionTreeClassifier(random_state=0))]),
    "RandomForest": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", RandomForestClassifier(random_state=0))]),
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", SVC())]),
}


In [ ]:
rows = []
for name, pipe in pipelines.items():
    t0 = time.time()
    pipe.fit(X_train, y_train)
    train_time = time.time() - t0
    pred = pipe.predict(X_test)
    rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, average="weighted"),
        "recall": recall_score(y_test, pred, average="weighted"),
        "f1": f1_score(y_test, pred, average="weighted"),
        "train_time_s": round(train_time, 2),
    })

scores = pd.DataFrame(rows).sort_values("f1", ascending=False).reset_index(drop=True)
scores.round(3).to_string(index=False)


### Confusion matrices

Rows = actual segment, columns = predicted. The diagonal is correct predictions — bright diagonal and pale everything else is what we want.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for ax, (name, pipe) in zip(axes.ravel(), pipelines.items()):
    ConfusionMatrixDisplay.from_estimator(
        pipe, X_test, y_test, ax=ax, cmap="Blues",
        xticks_rotation=25, colorbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.show()


### Cross-validation

5 stratified folds, F1 (weighted) per fold. Training 5 times on different slices shows whether a score was luck of the split — tight boxes = stable model.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
fold_scores = {}
for name, pipe in pipelines.items():
    fold_scores[name] = cross_val_score(
        pipe, X_train, y_train, cv=cv, scoring="f1_weighted")

cv_df = pd.DataFrame({
    "model": list(fold_scores),
    "cv_f1_mean": [s.mean() for s in fold_scores.values()],
    "cv_f1_std": [s.std() for s in fold_scores.values()],
}).sort_values("cv_f1_mean", ascending=False)
print(cv_df.round(3).to_string(index=False))

cv_long = pd.DataFrame(fold_scores).melt(var_name="model", value_name="f1")
plt.figure(figsize=(8, 5))
sns.boxplot(data=cv_long, x="model", y="f1", hue="model",
            legend=False, palette="Set2")
plt.ylim(0.85, 1.0)
plt.title("5-fold cross-validation — F1 per fold")
plt.xlabel("")
plt.ylabel("F1 (weighted)")
plt.tight_layout()
plt.show()


### GridSearchCV on the full pipelines

`GridSearchCV` wraps the whole pipeline with `cv=5`, so the scaler is re-fitted on every fold — never once upfront (that would leak). It refits the best configuration on the full training set.

In [ ]:
grids = {
    "LogisticRegression": {"classifier__C": [0.01, 0.1, 1, 10]},
    "DecisionTree": {"classifier__max_depth": [None, 5, 10, 20]},
    "RandomForest": {"classifier__n_estimators": [100, 200],
                     "classifier__max_depth": [None, 10, 20]},
    "SVM": {"classifier__C": [0.1, 1, 10]},
}

best = {}
rows = []
for name, pipe in pipelines.items():
    gs = GridSearchCV(pipe, grids[name], cv=cv, scoring="f1_weighted")
    gs.fit(X_train, y_train)
    best[name] = gs
    rows.append({
        "model": name,
        "best_params": str(gs.best_params_).replace("classifier__", ""),
        "cv_f1": round(gs.best_score_, 3),
        "test_f1": round(f1_score(y_test, gs.predict(X_test), average="weighted"), 3),
    })

gs_df = pd.DataFrame(rows).sort_values("test_f1", ascending=False).reset_index(drop=True)
gs_df.to_string(index=False)


In [ ]:
plt.figure(figsize=(7, 4))
sns.barplot(data=gs_df, x="model", y="test_f1", hue="model",
            legend=False, palette="Set2")
plt.ylim(0.8, 1.0)
plt.title("Test F1 after GridSearchCV")
plt.xlabel("")
plt.ylabel("F1 (weighted)")
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()


In [ ]:
rf_best = best["RandomForest"].best_estimator_
lr_best = best["LogisticRegression"].best_estimator_

imp = pd.Series(
    rf_best.named_steps["classifier"].feature_importances_, index=feats)
coef = pd.Series(
    np.abs(lr_best.named_steps["classifier"].coef_).mean(axis=0), index=feats)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
imp.sort_values().plot.barh(ax=axes[0], color="steelblue")
axes[0].set_title("Random Forest — feature importance")
axes[0].set_xlabel("importance")
coef.sort_values().plot.barh(ax=axes[1], color="darkorange")
axes[1].set_title("Logistic Regression — mean |coefficient|")
axes[1].set_xlabel("mean |coefficient|")
plt.tight_layout()
plt.show()


In [ ]:
winner_name = gs_df.iloc[0]["model"]
winner = best[winner_name].best_estimator_
print("best model:", winner_name)

Path("../../models").mkdir(exist_ok=True)

for name, gs in best.items():
    joblib.dump(gs.best_estimator_, f"../../models/{name}.joblib")
    print("saved:", name + ".joblib")

joblib.dump(winner, "../../models/pipeline.joblib")
print("saved: pipeline.joblib ->", winner_name)

reloaded = joblib.load("../../models/pipeline.joblib")
sample = X_test.iloc[[0]]
print("reloaded pipeline predicts:", reloaded.predict(sample)[0])
print("actual segment:            ", y_test.iloc[0])


### Findings

- **RandomForest wins** (test F1 ≈ 0.97, Logistic Regression ≈ 0.90) — expected: the boundaries between segments are non-linear, which trees capture and a linear model cannot.
- Scores are high because `target` *is* the KMeans output — these models learn to reproduce the clustering, not an external ground truth.
- `models/` holds every trained pipeline (`LogisticRegression.joblib`, `DecisionTree.joblib`, `RandomForest.joblib`, `SVM.joblib`) plus `pipeline.joblib` — the winner, scaler + classifier with string class names. **FS7's Streamlit app loads `pipeline.joblib`** and calls `pipeline.predict(raw_row)`; preprocessing is never re-coded.
- SMOTE (`imblearn`) was applied to the training set right before training, as the brief's class-imbalance option — scores stayed ~unchanged, which is expected when classes already sit at 38 / 35 / 27%.
- FS6 wraps the same trainings in `mlflow.start_run()` to track params, metrics and the model itself.